In [2]:
!pip install groq

In [5]:
"""
classify_prompts_groq.py
------------------------
Classifies each unique journal prompt as REFLECTIVE or REDIRECTIVE
using the Groq API (llama-3.3-70b-versatile).

Reads the enriched input file (which already contains all prior columns:
participant_id, week, date, prompt_text, journal_response_text,
behavioral_domain_category, signal_before, signal_after,
signal_change, behavioral_improvement).

Adds a new column: prompt_type (REFLECTIVE / REDIRECTIVE)

Writes two output files — both preserve ALL existing columns:
  - prompts_classified.csv         : full table + prompt_type
  - prompts_redirective_only.csv   : filtered to REDIRECTIVE rows only

Install dependency:
    pip install groq pandas

Usage:
    export GROQ_API_KEY="your_key_here"
    python classify_prompts_groq.py
"""

import json
import os
import time
from pathlib import Path

import pandas as pd
from groq import Groq

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_CSV    = "prompts_with_signals_task3_3day.csv"    # ← enriched input file
OUT_FULL     = "prompts_classified_task4_3day.csv"            # all rows + prompt_type
# OUT_FILTERED = "prompts_redirective_only_task4_n.csv"      # REDIRECTIVE rows only
CACHE_FILE   = "classification_cache_n.json"         # skip already-done prompts

MODEL        = "llama-3.3-70b-versatile"
BATCH_SAVE   = 50      # save cache every N new API calls
RETRY_LIMIT  = 3       # retries per prompt on API error
RETRY_DELAY  = 5       # seconds between retries

# Placeholder values with no meaningful prompt text — label as None
SKIP_VALUES  = {"No manual key", "No auto key", "", "failure"}

# ── Classification prompt ─────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are classifying AI journaling prompts into two types.

REFLECTIVE: The prompt asks the user to reflect on or make meaning of existing
behavior. It observes something and asks how the user feels about it or what it
means to them.
Example: "Your running routine has really taken off! How's that influencing your day?"

REDIRECTIVE: The prompt explicitly surfaces a behavioral observation AND suggests
or nudges toward a behavior change. It implies something could or should be different.
Example: "Consider the impact of less walking and more screen time on your well-being.
Could increasing movement lighten your mood?"

Respond with only one word: REFLECTIVE or REDIRECTIVE"""

def build_user_message(prompt_text: str) -> str:
    return (
        f'Prompt to classify: "{prompt_text}"\n'
        f'Respond with only one word: REFLECTIVE or REDIRECTIVE'
    )


# ── Load input ────────────────────────────────────────────────────────────────
print(f"Loading: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)
print(f"  Shape           : {df.shape}")
print(f"  Columns         : {df.columns.tolist()}")

# Separate rows that have classifiable prompt text from placeholders
mask_skip = df['prompt_text'].isin(SKIP_VALUES) | df['prompt_text'].isna()
df_classifiable = df[~mask_skip].copy()
df_skip         = df[mask_skip].copy()

unique_prompts = df_classifiable['prompt_text'].drop_duplicates().tolist()
print(f"  Classifiable rows : {len(df_classifiable)}")
print(f"  Skipped rows      : {len(df_skip)}")
print(f"  Unique prompts    : {len(unique_prompts)}")


# ── Load cache (re-run safe) ──────────────────────────────────────────────────
cache_path = Path(CACHE_FILE)
cache: dict = json.loads(cache_path.read_text()) if cache_path.exists() else {}
already_done = sum(1 for p in unique_prompts if p in cache)
print(f"  Already cached  : {already_done} / {len(unique_prompts)}")


# ── Groq client ───────────────────────────────────────────────────────────────
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))


# ── Classify ──────────────────────────────────────────────────────────────────
def classify_prompt(prompt_text: str) -> str:
    """Return REFLECTIVE or REDIRECTIVE via Groq API with retry logic."""
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": build_user_message(prompt_text)},
                ],
                max_tokens=20,
                temperature=0.0,
            )
            label = response.choices[0].message.content.strip().upper()
            # Accept partial match in case model adds punctuation
            if "REDIRECTIVE" in label:
                return "REDIRECTIVE"
            return "REFLECTIVE"
        except Exception as e:
            print(f"    Attempt {attempt}/{RETRY_LIMIT} failed: {e}")
            if attempt < RETRY_LIMIT:
                time.sleep(RETRY_DELAY)
    return "REFLECTIVE"   # safe fallback after all retries exhausted


print("\nClassifying prompts via Groq...")
to_classify = [p for p in unique_prompts if p not in cache]
total = len(to_classify)

for i, prompt_text in enumerate(to_classify, start=1):
    cache[prompt_text] = classify_prompt(prompt_text)

    if i % BATCH_SAVE == 0 or i == total:
        cache_path.write_text(json.dumps(cache, indent=2))
        print(f"  [{i:>4}/{total}] saved cache...")

print(f"\nClassification complete. Total cache size: {len(cache)}")


# ── Map labels back onto the full dataframe ───────────────────────────────────
# Classifiable rows get label from cache
df_classifiable['prompt_type'] = df_classifiable['prompt_text'].map(cache)

# Skipped/placeholder rows get None
df_skip = df_skip.copy()
df_skip['prompt_type'] = None

# Reconstruct in original row order, preserving ALL prior columns
df_out = pd.concat([df_classifiable, df_skip]).sort_index()


# ── Summary ───────────────────────────────────────────────────────────────────
counts = df_out['prompt_type'].value_counts()
print("\n── Classification results ──────────────────────────────────")
print(f"  REFLECTIVE  : {counts.get('REFLECTIVE',  0)}")
print(f"  REDIRECTIVE : {counts.get('REDIRECTIVE', 0)}")
print(f"  Unclassified: {df_out['prompt_type'].isna().sum()}")

print("\nBreakdown by behavioral domain:")
by_cat = (
    df_out[df_out['prompt_type'].notna()]
    .groupby('behavioral_domain_category')['prompt_type']
    .value_counts()
    .unstack(fill_value=0)
)
print(by_cat)


# ── Export ────────────────────────────────────────────────────────────────────
# Full table — all original columns + prompt_type
df_out.to_csv(OUT_FULL, index=False)
print(f"\nSaved full table        → {OUT_FULL}  ({len(df_out)} rows, {len(df_out.columns)} columns)")

# REDIRECTIVE-only — same columns, filtered rows
# df_redirective = df_out[df_out['prompt_type'] == 'REDIRECTIVE']
# df_redirective.to_csv(OUT_FILTERED, index=False)
# print(f"Saved REDIRECTIVE only  → {OUT_FILTERED}  ({len(df_redirective)} rows)")

Loading: prompts_with_signals_task3_3day.csv
  Shape           : (369, 13)
  Columns         : ['participant_id', 'week', 'date', 'prompt_text', 'journal_response_text', 'behavioral_domain_category', 'signal_before', 'signal_after', 'signal_change', 'behavioral_improvement', 'n_features_total', 'n_features_improved', 'baseline_imputed']
  Classifiable rows : 369
  Skipped rows      : 0
  Unique prompts    : 353
  Already cached  : 353 / 353

Classifying prompts via Groq...

Classification complete. Total cache size: 353

── Classification results ──────────────────────────────────
  REFLECTIVE  : 268
  REDIRECTIVE : 101
  Unclassified: 0

Breakdown by behavioral domain:
prompt_type                 REDIRECTIVE  REFLECTIVE
behavioral_domain_category                         
digital_habits                       28         100
physical_fitness                     17          68
sleep                                25          13
social_interaction                   31          87

Saved fu